# Tideway Data API Examples

Data search, node lookups, kinds, and partitions via `tideway.data`. Reference: https://docs.bmc.com/xwiki/bin/view/IT-Operations-Management/Discovery/BMC-Helix-Discovery/DAAS/Integrating/Using-the-REST-APIs/Endpoints-in-the-REST-API/.


## Setup
- Install tideway (e.g. `pip install -e .` from repo root).
- Set `TARGET` and `TOKEN` below. Do **not** commit secrets.
- Adjust queries/endpoints to match your appliance.


In [ ]:
import tideway
from urllib.parse import quote

# Configuration
TARGET = 'appliance-hostname-or-ip'  # e.g. 'discovery.example.com'
TOKEN = 'your-api-token'            # keep secrets out of source control
API_VERSION = '1.16'                # latest supported API
SSL_VERIFY = False                  # set to True when using valid certs

tw = tideway.appliance(TARGET, TOKEN, api_version=API_VERSION, ssl_verify=SSL_VERIFY)
data = tw.data()

def show_response(resp, label):
    if resp.ok:
        try:
            return resp.json()
        except Exception:
            return resp.text
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    return {'endpoint': label, 'status': resp.status_code, 'body': body}

tw.api_about.json()  # quick sanity check


## Data search endpoints
- Use TPL search syntax for `/data/search`.
- `format` supports `object`, `tree`, or default tabular rows.
- BMC doc reference above for query capabilities.


In [ ]:
# GET /data/search with query string
search_query = "search Host show name, os_type, ip_addr"  # edit query per docs
search_query_encoded = quote(search_query)

helper_search = data.search(search_query, format="object", limit=50, bulk=False)
direct_search = data.get(f"/data/search?query={search_query_encoded}&format=object&limit=50")

search_calls = {
    "helper": show_response(helper_search, "/data/search?format=object"),
    "direct_get": show_response(direct_search, "/data/search (GET)"),
}
search_calls


In [ ]:
# Bulk search to iterate through paginated results
bulk_results = data.search_bulk(search_query, format="object", limit=200, record_limit=200)

if hasattr(bulk_results, "ok"):
    bulk_preview = show_response(bulk_results, "/data/search bulk")
else:
    bulk_preview = bulk_results[:5]  # preview first rows/objects

direct_first_page = data.get(f"/data/search?query={search_query_encoded}&format=object&limit=200")

bulk_calls = {
    "helper_bulk_preview": bulk_preview,
    "direct_first_page": show_response(direct_first_page, "/data/search (GET, first page)"),
}
bulk_calls


## POST /data/search (long queries or alternate formats)
Use POST when the search string is long or when sending JSON payloads.


In [ ]:
long_query_body = {
    "query": "search SoftwareInstance where type matches \"Database\" show name, version, host.name"
}

helper_long_query = data.search(long_query_body, format="tree", limit=50, bulk=False)
direct_long_query = data.post('/data/search', long_query_body)

long_query_calls = {
    "/data/search (helper, tree)": show_response(helper_long_query, "/data/search (POST, tree)"),
    "/data/search (direct POST)": show_response(direct_long_query, "/data/search (POST)"),
}
long_query_calls


## Condition endpoints
- Discover available condition parameters/templates first.
- Fill `condition_body` with a condition and attributes per the API schema.


In [ ]:
condition_params = data.get_data_condition_params()
condition_templates = data.get_data_condition_templates

direct_condition_params = data.get('/data/condition/params')
direct_condition_templates = data.get('/data/condition/templates')

condition_meta = {
    'helpers': {
        '/data/condition/params': show_response(condition_params, '/data/condition/params'),
        '/data/condition/templates': show_response(condition_templates, '/data/condition/templates'),
    },
    'direct': {
        '/data/condition/params (GET)': show_response(direct_condition_params, '/data/condition/params'),
        '/data/condition/templates (GET)': show_response(direct_condition_templates, '/data/condition/templates'),
    }
}
condition_meta


In [ ]:
condition_body = {
    # Example shape - adjust to your condition
    # "condition": {"attribute": "Host:os", "operator": "=", "value": "Linux"},
    # "attributes": ["name", "os", "serial_number"],
    # "parameters": {},
}

if condition_body:
    condition_results_helper = data.post_data_condition(condition_body)
    condition_results_direct = data.post('/data/condition', condition_body)
    condition_results = {
        'helper': show_response(condition_results_helper, "/data/condition (helper)"),
        'direct': show_response(condition_results_direct, "/data/condition (direct POST)"),
    }
else:
    condition_results = "Set condition_body to post /data/condition"
condition_results


## Node lookups and graphs
Set `node_id` to inspect node state and graph relationships.


In [ ]:
node_id = ''  # e.g. '1234567890abcdef'

if node_id:
    node_details_helper = data.get_data_nodes(node_id, relationships=True)
    node_graph_helper = data.get_data_nodes_graph(node_id, focus="software-connected", complete=False)

    node_details_direct = data.get(f"/data/nodes/{node_id}?relationships=true")
    node_graph_direct = data.get(f"/data/nodes/{node_id}/graph?focus=software-connected&complete=false")

    node_info = {
        f"/data/nodes/{node_id} (helper)": show_response(node_details_helper, f"/data/nodes/{node_id}?relationships=true"),
        f"/data/nodes/{node_id}/graph (helper)": show_response(node_graph_helper, f"/data/nodes/{node_id}/graph"),
        f"/data/nodes/{node_id} (direct)": show_response(node_details_direct, f"/data/nodes/{node_id}?relationships=true"),
        f"/data/nodes/{node_id}/graph (direct)": show_response(node_graph_direct, f"/data/nodes/{node_id}/graph"),
    }
else:
    node_info = "Set node_id to inspect node details."
node_info


## Node kinds and attribute values
Use these to inspect specific node kinds and enumerate attribute values.


In [ ]:
kind = 'Host'      # edit node kind per docs
attribute = 'os'   # attribute to enumerate values for

kind_resp_helper = data.get_data_kinds(kind, format="object", limit=25)
kind_values_resp_helper = data.get_data_kinds_values(kind, attribute)

kind_resp_direct = data.get(f"/data/kinds/{kind}?format=object&limit=25")
kind_values_resp_direct = data.get(f"/data/kinds/{kind}/values/{attribute}")

kind_calls = {
    f"/data/kinds/{kind} (helper)": show_response(kind_resp_helper, f"/data/kinds/{kind}?format=object"),
    f"/data/kinds/{kind} (direct)": show_response(kind_resp_direct, f"/data/kinds/{kind}?format=object&limit=25"),
    f"/data/kinds/{kind}/values/{attribute} (helper)": show_response(kind_values_resp_helper, f"/data/kinds/{kind}/values/{attribute}"),
    f"/data/kinds/{kind}/values/{attribute} (direct)": show_response(kind_values_resp_direct, f"/data/kinds/{kind}/values/{attribute}"),
}
kind_calls


## Partitions, candidates, and external consumers
GET calls are safe to run; POST bodies are placeholders until filled.


In [ ]:
partitions = data.get_data_partitions
external_consumers = data.get_data_external_consumers

direct_partitions = data.get('/data/partitions')
direct_external_consumers = data.get('/data/external_consumers')

data_admin_calls = {
    '/data/partitions (helper)': show_response(partitions, '/data/partitions'),
    '/data/partitions (direct)': show_response(direct_partitions, '/data/partitions'),
    '/data/external_consumers (helper)': show_response(external_consumers, '/data/external_consumers'),
    '/data/external_consumers (direct)': show_response(direct_external_consumers, '/data/external_consumers'),
}
data_admin_calls


In [ ]:
partition_body = {
    # 'name': 'example_partition',
    # 'label': 'Example partition description',
}
candidate_body = {
    # 'ip': '10.0.0.1',
    # 'dns_name': 'host.example.com',
}
external_consumer_body = {
    # 'name': 'consumer_id',
    # 'endpoint': 'https://example.com/ingest',
}

post_responses = {}
if partition_body:
    created_partition_helper = data.post_data_partitions(partition_body)
    created_partition_direct = data.post('/data/partitions', partition_body)
    post_responses['/data/partitions (helper POST)'] = show_response(created_partition_helper, '/data/partitions (POST)')
    post_responses['/data/partitions (direct POST)'] = show_response(created_partition_direct, '/data/partitions (POST)')

if candidate_body:
    best_candidate_helper = data.post_data_candidate(candidate_body)
    best_candidate_direct = data.post('/data/candidate', candidate_body)
    post_responses['/data/candidate (helper POST)'] = show_response(best_candidate_helper, '/data/candidate')
    post_responses['/data/candidate (direct POST)'] = show_response(best_candidate_direct, '/data/candidate')

if external_consumer_body:
    created_consumer_helper = data.post_data_external_consumer(external_consumer_body)
    created_consumer_direct = data.post('/data/external_consumers', external_consumer_body)
    post_responses['/data/external_consumers (helper POST)'] = show_response(created_consumer_helper, '/data/external_consumers')
    post_responses['/data/external_consumers (direct POST)'] = show_response(created_consumer_direct, '/data/external_consumers')

post_responses or "Set request bodies to run POST examples."
